# 🎙️ ZeroTTS Studio - Google Colab
[![GitHub Repo](https://img.shields.io/badge/GitHub-RevenantKitana%2FTSS-181717?logo=github)](https://github.com/RevenantKitana/TSS)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RevenantKitana/TSS/blob/main/ZeroTTS_Colab_FreeTier.ipynb)

> **Lưu ý:** Chọn **Runtime** -> **Change runtime type** -> **T4 GPU** trước khi chạy.


In [ ]:
#@title Bước 1: Kết nối Google Drive & Kiểm tra Phần Cứng { run: "auto", display-mode: "form" }
MOUNT_GOOGLE_DRIVE = True #@param {type:"boolean"}
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/ZeroTTS_Outputs"
DRIVE_INPUT_DIR = "/content/drive/MyDrive/ZeroTTS_Inputs"

import os
import subprocess

# 1. Kiểm tra GPU
print("🔍 Đang kiểm tra phần cứng...")
try:
    gpu_info = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"]).decode("utf-8").strip()
    print(f"✅ Phát hiện GPU: {gpu_info}")
    HAS_GPU = True
except Exception:
    print("ℹ️ Đang chạy trên CPU. Bạn có thể bật GPU tại: Runtime -> Change runtime type -> T4 GPU.")
    HAS_GPU = False

# 2. Mount Google Drive nếu được bật
if MOUNT_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
        os.makedirs(DRIVE_INPUT_DIR, exist_ok=True)
        os.environ["ZEROTTS_OUTPUT_DIR"] = DRIVE_OUTPUT_DIR
        os.environ["ZEROTTS_INPUT_DIR"] = DRIVE_INPUT_DIR
        print(f"📁 Google Drive đã kết nối thành công:")
        print(f"   -> Thư mục lưu audio:    {DRIVE_OUTPUT_DIR}")
        print(f"   -> Thư mục lưu kịch bản: {DRIVE_INPUT_DIR}")
    except Exception as e:
        print(f"⚠️ Không thể kết nối Google Drive ({e}). Sử dụng thư mục tạm /content/outputs và /content/ZeroTTS_Inputs.")
        os.environ["ZEROTTS_OUTPUT_DIR"] = "/content/outputs"
        os.environ["ZEROTTS_INPUT_DIR"] = "/content/ZeroTTS_Inputs"
else:
    os.environ["ZEROTTS_OUTPUT_DIR"] = "/content/outputs"
    os.environ["ZEROTTS_INPUT_DIR"] = "/content/ZeroTTS_Inputs"
    print("📁 Dữ liệu sẽ lưu tạm tại /content (sẽ bị xoá khi tắt Colab).")


In [ ]:
#@title Bước 2: Nạp Mã Nguồn Từ GitHub { display-mode: "form" }
import os
import sys
import subprocess
import shutil

REPO_URL = "https://github.com/RevenantKitana/TSS.git"
APP_DIR = "/content/TSS"

print(f"📥 Đang tải mã nguồn từ: {REPO_URL}...")
if not os.path.exists(os.path.join(APP_DIR, ".git")):
    shutil.rmtree(APP_DIR, ignore_errors=True)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, APP_DIR], check=True)
    print("✅ Clone mã nguồn thành công từ RevenantKitana/TSS!")
else:
    print("🔄 Cập nhật mã nguồn mới nhất từ GitHub...")
    subprocess.run(["git", "-C", APP_DIR, "fetch", "--all"], check=True)
    subprocess.run(["git", "-C", APP_DIR, "reset", "--hard", "origin/main"], check=True)
    print("✅ Đã đồng bộ mã nguồn mới nhất từ RevenantKitana/TSS!")

# Tải Git LFS objects cho các gói giọng trong Voice_ZeroTTS_model
print("📦 Đang đồng bộ Git LFS (Voice embeddings & preview clips)...")
try:
    if shutil.which("git-lfs"):
        subprocess.run(["git", "lfs", "install"], check=False)
        subprocess.run(["git", "-C", APP_DIR, "lfs", "pull"], check=False)
except Exception:
    pass

# Copy đồng bộ vào /content/webui, /content/src và /content/Voice_ZeroTTS_model
for folder in ["webui", "src", "Voice_ZeroTTS_model"]:
    src_f = os.path.join(APP_DIR, folder)
    dst_f = os.path.join("/content", folder)
    if os.path.exists(src_f):
        try:
            shutil.copytree(src_f, dst_f, dirs_exist_ok=True)
        except Exception:
            pass

for p in [APP_DIR, os.path.join(APP_DIR, "src"), os.path.join(APP_DIR, "webui"), "/content", "/content/src", "/content/webui"]:
    if os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p)

%cd $APP_DIR

In [ ]:
#@title Bước 3: Cài Đặt Thư Viện { display-mode: "form" }
print("🔧 Đang cài đặt thư viện hệ thống và Python (chỉ mất ~1 phút)...")
!apt-get update -qq && apt-get install -y -qq ffmpeg libportaudio2 git-lfs

# 1. Cài đặt các gói phụ trợ
!pip install -q soundfile sounddevice fastapi uvicorn tokenizers huggingface_hub scipy requests pydantic
!pip install -q --no-deps -e .

# 2. Cài đặt ONNX Runtime GPU (Bản CUDA 12 tương thích hoàn hảo Colab T4)
if HAS_GPU:
    print("⚡ Đang cài đặt ONNX Runtime GPU (CUDA 12 T4)...\n")
    !pip uninstall -y -q onnxruntime onnxruntime-gpu 2>/dev/null || true
    !pip install -q "onnxruntime-gpu>=1.19.0,<=1.20.1"
    # Cấu hình dynamic linker hệ thống nhận diện toàn bộ thư viện CUDA & cuDNN từ pip và hệ thống
    !mkdir -p /etc/ld.so.conf.d/
    !find /usr/local/lib/python* /usr/lib/python* /usr/lib64-nvidia /usr/local/cuda* -name "lib" -type d -path "*nvidia*" 2>/dev/null > /etc/ld.so.conf.d/nvidia-libs.conf
    !echo "/usr/lib64-nvidia" >> /etc/ld.so.conf.d/nvidia-libs.conf
    !echo "/usr/local/cuda/lib64" >> /etc/ld.so.conf.d/nvidia-libs.conf
    !ldconfig 2>/dev/null || true
else:
    print("⚙️ Đang cài đặt ONNX Runtime CPU...")
    !pip uninstall -y -q onnxruntime onnxruntime-gpu 2>/dev/null || true
    !pip install -q "onnxruntime>=1.17.0"

# 3. Kiểm tra và Preload CUDA
import os, sys
try:
    import torch
except Exception:
    pass

for mod in list(sys.modules.keys()):
    if "onnxruntime" in mod:
        del sys.modules[mod]
import onnxruntime as ort

if hasattr(ort, 'preload_dlls'):
    try:
        ort.preload_dlls()
    except Exception:
        pass

avail = ort.get_available_providers()
print(f"\n🔍 Kết quả kiểm tra ONNX Providers: {avail}")
if "CUDAExecutionProvider" in avail:
    print("🚀 [GPU T4] ĐÃ KÍCH HOẠT GPU THÀNH CÔNG! (CUDAExecutionProvider sẵn sàng)")
elif HAS_GPU:
    print("⚠️ Lưu ý: Nếu vừa cài đặt lại, hãy vào menu: Runtime -> Restart session (Khởi động lại phiên) rồi bấm Chạy lại Bước 3.")
else:
    print("ℹ️ Đang chạy trên CPU (CPUExecutionProvider).")

print("\n✅ Cài đặt hoàn tất! Toàn bộ tính năng đã sẵn sàng.")


In [ ]:
#@title Bước 4: Tải Model Weights & Gói Giọng { display-mode: "form" }
from huggingface_hub import snapshot_download
import os
import sys
import subprocess

try:
    import torch
except Exception:
    pass

import onnxruntime as ort
if hasattr(ort, 'preload_dlls'):
    try:
        ort.preload_dlls()
    except Exception:
        pass

avail_p = ort.get_available_providers()
print(f"🔍 ONNX Providers khả dụng: {avail_p}")
if "CUDAExecutionProvider" in avail_p:
    print("🚀 [GPU] CUDAExecutionProvider khả dụng!")
else:
    print("⚙️ [CPU] Đang dùng CPUExecutionProvider.")

APP_DIR = "/content/TSS"
MODEL_DIR = os.path.join(APP_DIR, "ZeroTTS_model") if os.path.exists(APP_DIR) else "/content/ZeroTTS_model"
os.environ["ZEROTTS_MODEL"] = MODEL_DIR

# Định vị thư mục giọng từ Voice_ZeroTTS_model
candidate_voices = [
    os.path.join(APP_DIR, "Voice_ZeroTTS_model", "voices"),
    os.path.join(APP_DIR, "Voice_ZeroTTS_model"),
    "/content/Voice_ZeroTTS_model/voices",
    "/content/Voice_ZeroTTS_model",
]
VOICES_DIR = next((p for p in candidate_voices if os.path.exists(p)), None)

print("📥 Đang tải weights mô hình ZeroTTS (~500MB)...")
snapshot_download(
    repo_id="zeroweight-ai/ZeroTTS",
    local_dir=MODEL_DIR,
    local_dir_use_symlinks=False
)
print(f"✅ Đã tải xong Model Weights tại: {MODEL_DIR}")

# Kiểm tra 8 gói giọng mẫu
if VOICES_DIR and os.path.exists(VOICES_DIR):
    voices = [f for f in os.listdir(VOICES_DIR) if not f.startswith(".") and os.path.isdir(os.path.join(VOICES_DIR, f))]
    print(f"🗣️ Đã xác thực {len(voices)} gói giọng sẵn có trong Voice_ZeroTTS_model: {voices}")
else:
    print("ℹ️ Đang sử dụng gói giọng mặc định của mô hình.")

for p in [APP_DIR, os.path.join(APP_DIR, "src"), os.path.join(APP_DIR, "webui"), "/content", "/content/src", "/content/webui"]:
    if os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p)

import webui.engine as engine
engine.set_model(MODEL_DIR, voices_dir=VOICES_DIR)
# Khởi tạo nạp model vào GPU ngay tại Bước 4 để xác thực và làm ấm bộ nhớ
_ = engine.get_tts()
print(f"⚡ ZeroTTS Engine đã nạp sẵn sàng!")


In [ ]:
#@title Bước 5: Khởi Chạy WebUI Studio (Cloudflare Tunnel) { display-mode: "form" }
import subprocess
import time
import re
import os
import sys
import glob
from IPython.display import display, HTML

# 1. Tự động xác định chính xác APP_DIR, MODEL_DIR và VOICES_DIR
if "APP_DIR" not in globals() or not APP_DIR:
    APP_DIR = "/content/TSS"

if "MODEL_DIR" not in globals() or not MODEL_DIR:
    model_candidates = [
        os.path.join(APP_DIR, "ZeroTTS_model"),
        "/content/TSS/ZeroTTS_model",
        "/content/ZeroTTS_model",
    ]
    MODEL_DIR = None
    for mc in model_candidates:
        if os.path.exists(os.path.join(mc, "config.json")):
            MODEL_DIR = mc
            break
    if not MODEL_DIR:
        MODEL_DIR = os.path.join(APP_DIR, "ZeroTTS_model")

# Tự động tải weights nếu chưa có
if not os.path.exists(os.path.join(MODEL_DIR, "config.json")):
    print(f"⏳ Chưa tìm thấy trọng số mô hình. Đang tự động tải ZeroTTS vào {MODEL_DIR}...")
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id="zeroweight-ai/ZeroTTS", local_dir=MODEL_DIR)
    print("✅ Đã tải xong Model Weights!")

# Tự động định vị thư mục giọng từ Voice_ZeroTTS_model
if "VOICES_DIR" not in globals() or not VOICES_DIR:
    for vc in [
        os.path.join(APP_DIR, "Voice_ZeroTTS_model", "voices"),
        os.path.join(APP_DIR, "Voice_ZeroTTS_model"),
        "/content/TSS/Voice_ZeroTTS_model/voices",
        "/content/Voice_ZeroTTS_model/voices",
    ]:
        if os.path.exists(vc):
            VOICES_DIR = vc
            break

if VOICES_DIR:
    os.environ["ZEROTTS_VOICES_DIR"] = VOICES_DIR
    print(f"🎙️ Nạp mẫu giọng từ: {VOICES_DIR}")

# 2. Tự động xác định chính xác vị trí webui/server.py và project root
server_candidates = [
    os.path.join(APP_DIR, "webui", "server.py"),
    "/content/TSS/webui/server.py",
    "/content/webui/server.py",
    os.path.join(os.getcwd(), "webui", "server.py"),
]
SERVER_SCRIPT = None
PROJECT_ROOT = None
for cand in server_candidates:
    if os.path.isfile(cand):
        SERVER_SCRIPT = os.path.abspath(cand)
        PROJECT_ROOT = os.path.dirname(os.path.dirname(SERVER_SCRIPT))
        break

if not SERVER_SCRIPT:
    matches = glob.glob("/content/**/webui/server.py", recursive=True) + glob.glob("**/webui/server.py", recursive=True)
    if matches:
        SERVER_SCRIPT = os.path.abspath(matches[0])
        PROJECT_ROOT = os.path.dirname(os.path.dirname(SERVER_SCRIPT))

if not SERVER_SCRIPT:
    raise FileNotFoundError("❌ Không tìm thấy file webui/server.py! Hãy chạy lại Bước 2 (Nạp mã nguồn từ GitHub [RevenantKitana/TSS](https://github.com/RevenantKitana/TSS)).")
print(f"🎯 Đã định vị WebUI Server tại: {SERVER_SCRIPT}")
print(f"📂 Project Root: {PROJECT_ROOT}")
os.chdir(PROJECT_ROOT)

# 3. Download cloudflared binary nếu chưa có
if not os.path.exists("/usr/local/bin/cloudflared"):
    print("📥 Đang tải Cloudflare Tunnel (cloudflared)...")
    !wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared

# 4. Chạy FastAPI Server ở chế độ nền
print("🚀 1. Đang khởi động ZeroTTS WebUI Server...")
server_cmd = [sys.executable, SERVER_SCRIPT, "--model", MODEL_DIR, "--host", "0.0.0.0", "--port", "7860"]
if VOICES_DIR:
    server_cmd.extend(["--voices", VOICES_DIR])
server_proc = subprocess.Popen(
    server_cmd,
    cwd=PROJECT_ROOT,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

time.sleep(3)

# 5. Khởi động Cloudflare Tunnel
print("🌐 2. Đang mở đường hầm Cloudflare Tunnel...")
tunnel_cmd = ["/usr/local/bin/cloudflared", "tunnel", "--url", "http://127.0.0.1:7860"]
tunnel_proc = subprocess.Popen(tunnel_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

public_url = None
for _ in range(50):
    line = tunnel_proc.stdout.readline()
    if not line:
        time.sleep(0.4)
        continue
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

print("=" * 70)
if public_url:
    display(HTML(f"""
    <div style="background: linear-gradient(135deg, #1e1e2e, #2d1f47); padding: 20px; border-radius: 12px; border: 1px solid #7c3aed; margin: 15px 0;">
        <h2 style="color: #a78bfa; margin: 0 0 10px 0;">🎉 ZeroTTS Studio Đã Sẵn Sàng!</h2>
        <p style="color: #e2e8f0; font-size: 15px; margin: 0 0 15px 0;">Mã nguồn: <a href="https://github.com/RevenantKitana/TSS" target="_blank" style="color: #38bdf8; text-decoration: underline;">RevenantKitana/TSS: Clone ZeroTSS</a></p>
        <a href="{public_url}" target="_blank" style="background: #7c3aed; color: #ffffff; padding: 12px 24px; border-radius: 8px; text-decoration: none; font-weight: bold; font-size: 16px; display: inline-block; box-shadow: 0 4px 14px rgba(124, 58, 237, 0.4); margin-top: 5px;">
            🚀 Mở WebUI Studio (Public URL)
        </a>
        <p style="color: #94a3b8; font-size: 13px; margin: 12px 0 0 0;">Link: <a href="{public_url}" target="_blank" style="color: #38bdf8;">{public_url}</a></p>
    </div>
    """))
else:
    print("⚠️ Đang khởi động tunnel, vui lòng kiểm tra lại log bên dưới.")
print("=" * 70 + "\n")

# Giữ tiến trình hoạt động và hiển thị log
try:
    while True:
        line = server_proc.stdout.readline()
        if line:
            print(line, end="")
        else:
            time.sleep(0.1)
except KeyboardInterrupt:
    print("\n🛑 Đã dừng WebUI Server.")
    server_proc.terminate()
    tunnel_proc.terminate()

In [ ]:
#@title Bước 5b: Tải Lên Kịch Bản (.docx / .txt) Vào Google Drive { display-mode: "form" }
import os
import sys
import shutil


APP_DIR = globals().get("APP_DIR") or "/content/TSS"
MODEL_DIR = globals().get("MODEL_DIR") or os.path.join(APP_DIR, "ZeroTTS_model")
VOICES_DIR = globals().get("VOICES_DIR") or None

if not VOICES_DIR:
    for vc in [
        os.path.join(APP_DIR, "Voice_ZeroTTS_model", "voices"),
        os.path.join(APP_DIR, "Voice_ZeroTTS_model"),
        "/content/TSS/Voice_ZeroTTS_model/voices" if True else "/kaggle/working/TSS/Voice_ZeroTTS_model/voices",
        "/content/Voice_ZeroTTS_model/voices" if True else "/kaggle/working/Voice_ZeroTTS_model/voices",
    ]:
        if os.path.exists(vc):
            VOICES_DIR = vc
            break

for p in [APP_DIR, os.path.join(APP_DIR, "src"), os.path.join(APP_DIR, "webui"), "/content" if True else "/kaggle/working", "/content/src" if True else "/kaggle/working/src", "/content/webui" if True else "/kaggle/working/webui"]:
    if os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p)

import webui.engine as engine

engine.set_model(MODEL_DIR, voices_dir=VOICES_DIR)

# Tự động phát hiện kịch bản từ file hoặc Google Drive / Thư mục làm việc
target_file = None
if INPUT_FILE_PATH.strip():
    cand = INPUT_FILE_PATH.strip()
    if os.path.exists(cand):
        target_file = cand
    else:
        in_dir = os.environ.get("ZEROTTS_INPUT_DIR", "/content/drive/MyDrive/ZeroTTS_Inputs" if True else "/kaggle/working/ZeroTTS_Inputs")
        cand_in_dir = os.path.join(in_dir, cand)
        if os.path.exists(cand_in_dir):
            target_file = cand_in_dir
        else:
            print(f"⚠️ Không tìm thấy '{cand}'. Sử dụng văn bản mẫu bên dưới.")
elif "LATEST_INPUT_FILE" in globals() and LATEST_INPUT_FILE and os.path.exists(LATEST_INPUT_FILE):
    target_file = LATEST_INPUT_FILE
    print(f"💡 Đang dùng tệp kịch bản vừa tải lên: {target_file}")
else:
    in_dir = os.environ.get("ZEROTTS_INPUT_DIR", "/content/drive/MyDrive/ZeroTTS_Inputs" if True else "/kaggle/working/ZeroTTS_Inputs")
    if os.path.exists(in_dir):
        files_in_dir = [os.path.join(in_dir, f) for f in os.listdir(in_dir) if f.lower().endswith((".docx", ".txt", ".md"))]
        if files_in_dir:
            files_in_dir.sort(key=lambda x: os.path.getmtime(x), reverse=True)
            target_file = files_in_dir[0]
            print(f"💡 Tự động nạp kịch bản mới nhất từ thư mục: {target_file}")

raw_script = sample_input_text
fallback_name = DEFAULT_PROJECT_NAME if "DEFAULT_PROJECT_NAME" in globals() else (PROJECT_NAME if "PROJECT_NAME" in globals() else "du_an_01")
if target_file and os.path.exists(target_file):
    print(f"📄 Đang đọc dữ liệu từ tệp: {target_file}...")
    raw_script, default_f = engine.read_input_file(target_file)
    if default_f:
        fallback_name = default_f

projects = engine.parse_multi_project_blocks(raw_script, default_name=fallback_name)
print(f"🎯 Nhận diện được {len(projects)} dự án trong chuỗi:")
for p in projects:
    print(f" 📁 Dự án: [{p['project_name']}] - Tổng số câu: {len(p['blocks'])}")
    for b in p['blocks']:
        st = "⏭️ BỎ QUA" if b['is_skipped'] else "✅ RENDER"
        print(f"    - [{b['tag']}]: {st} | {b['text'][:40]}...")

use_voice_flag = VOICE_NAME != "unconditional"
voice_param = None if VOICE_NAME == "unconditional" else VOICE_NAME

print(f"\n🔊 Đang bắt đầu quá trình tạo giọng nói với giọng [{VOICE_NAME}]...")
result_meta = {}
last_status = None
for status_msg, last_file, seg_text, q_meta in engine.generate_projects_queue_stream(
    text=raw_script,
    voice_name=voice_param,
    custom_name=fallback_name,
    auto_concat=AUTO_MERGE,
    merged_format=MERGED_FORMAT,
    cfg_scale=CFG_SCALE,
    audio_temperature=AUDIO_TEMP,
    use_voice=use_voice_flag,
    num_workers=int(NUM_WORKERS),
    result=result_meta,
):
    if status_msg and status_msg != last_status:
        last_status = status_msg
        print(f"   {status_msg}")

print(f"\n🎉 Hoàn thành toàn bộ kịch bản!")
all_folders = result_meta.get("folders", [])
for out_f in all_folders:
    if os.path.exists(out_f):
        print(f"\n📂 Thư mục xuất: {out_f}")
        for f in sorted(os.listdir(out_f)):
            fpath = os.path.join(out_f, f)
            size_kb = os.path.getsize(fpath) / 1024
            print(f"  ├── {f} ({size_kb:.1f} KB)")


In [ ]:
#@title Bước 6: Chạy Render Hàng Loạt Bằng Script (CLI) { display-mode: "form" }
import os
import sys
import shutil

INPUT_FILE_PATH = "" #@param {type:"string"}
sample_input_text = """$[An_toàn_nghiệp_vụ] // Thư mục 1
[Text 1]
Chào mừng các bạn đến với khóa đào tạo An toàn lao động. [pause: 1.5s]

[Text 2]
Hãy luôn trang bị đầy đủ đồ bảo hộ cá nhân khi vào khu vực công trường.

#[Bỏ qua]
Đoạn này nháp, hệ thống tự động bỏ qua không đọc.

$[Kỹ_năng_giao_tiếp] // Thư mục 2
[Text 1]
Giao tiếp hiệu quả và lắng nghe tích cực là chìa khóa thành công!
"""
VOICE_NAME = "maichi" #@param ["maichi", "baotrang", "kimoanh", "hamy", "giahuy", "huuduc", "quangminh", "tiendat", "unconditional"]
DEFAULT_PROJECT_NAME = "du_an_colab_01" #@param {type:"string"}
CFG_SCALE = 1.0 #@param {type:"slider", min:1.0, max:3.0, step:0.1}
AUDIO_TEMP = 0.8 #@param {type:"slider", min:0.3, max:1.2, step:0.05}
AUTO_MERGE = True #@param {type:"boolean"}
MERGED_FORMAT = "MP3" #@param ["MP3", "WAV", "FLAC", "M4A", "OGG"]
NUM_WORKERS = 1 #@param [1, 2, 3, 4, 6, 8] {type:"raw"}

APP_DIR = globals().get("APP_DIR") or "/content/TSS"
MODEL_DIR = globals().get("MODEL_DIR") or os.path.join(APP_DIR, "ZeroTTS_model")
VOICES_DIR = globals().get("VOICES_DIR") or None

if not VOICES_DIR:
    for vc in [
        os.path.join(APP_DIR, "Voice_ZeroTTS_model", "voices"),
        os.path.join(APP_DIR, "Voice_ZeroTTS_model"),
        "/content/TSS/Voice_ZeroTTS_model/voices" if True else "/kaggle/working/TSS/Voice_ZeroTTS_model/voices",
        "/content/Voice_ZeroTTS_model/voices" if True else "/kaggle/working/Voice_ZeroTTS_model/voices",
    ]:
        if os.path.exists(vc):
            VOICES_DIR = vc
            break

for p in [APP_DIR, os.path.join(APP_DIR, "src"), os.path.join(APP_DIR, "webui"), "/content" if True else "/kaggle/working", "/content/src" if True else "/kaggle/working/src", "/content/webui" if True else "/kaggle/working/webui"]:
    if os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p)

import webui.engine as engine

engine.set_model(MODEL_DIR, voices_dir=VOICES_DIR)

# Tự động phát hiện kịch bản từ file hoặc Google Drive / Thư mục làm việc
target_file = None
if INPUT_FILE_PATH.strip():
    cand = INPUT_FILE_PATH.strip()
    if os.path.exists(cand):
        target_file = cand
    else:
        in_dir = os.environ.get("ZEROTTS_INPUT_DIR", "/content/drive/MyDrive/ZeroTTS_Inputs" if True else "/kaggle/working/ZeroTTS_Inputs")
        cand_in_dir = os.path.join(in_dir, cand)
        if os.path.exists(cand_in_dir):
            target_file = cand_in_dir
        else:
            print(f"⚠️ Không tìm thấy '{cand}'. Sử dụng văn bản mẫu bên dưới.")
elif "LATEST_INPUT_FILE" in globals() and LATEST_INPUT_FILE and os.path.exists(LATEST_INPUT_FILE):
    target_file = LATEST_INPUT_FILE
    print(f"💡 Đang dùng tệp kịch bản vừa tải lên: {target_file}")
else:
    in_dir = os.environ.get("ZEROTTS_INPUT_DIR", "/content/drive/MyDrive/ZeroTTS_Inputs" if True else "/kaggle/working/ZeroTTS_Inputs")
    if os.path.exists(in_dir):
        files_in_dir = [os.path.join(in_dir, f) for f in os.listdir(in_dir) if f.lower().endswith((".docx", ".txt", ".md"))]
        if files_in_dir:
            files_in_dir.sort(key=lambda x: os.path.getmtime(x), reverse=True)
            target_file = files_in_dir[0]
            print(f"💡 Tự động nạp kịch bản mới nhất từ thư mục: {target_file}")

raw_script = sample_input_text
fallback_name = DEFAULT_PROJECT_NAME if "DEFAULT_PROJECT_NAME" in globals() else (PROJECT_NAME if "PROJECT_NAME" in globals() else "du_an_01")
if target_file and os.path.exists(target_file):
    print(f"📄 Đang đọc dữ liệu từ tệp: {target_file}...")
    raw_script, default_f = engine.read_input_file(target_file)
    if default_f:
        fallback_name = default_f

projects = engine.parse_multi_project_blocks(raw_script, default_name=fallback_name)
print(f"🎯 Nhận diện được {len(projects)} dự án trong chuỗi:")
for p in projects:
    print(f" 📁 Dự án: [{p['project_name']}] - Tổng số câu: {len(p['blocks'])}")
    for b in p['blocks']:
        st = "⏭️ BỎ QUA" if b['is_skipped'] else "✅ RENDER"
        print(f"    - [{b['tag']}]: {st} | {b['text'][:40]}...")

use_voice_flag = VOICE_NAME != "unconditional"
voice_param = None if VOICE_NAME == "unconditional" else VOICE_NAME

print(f"\n🔊 Đang bắt đầu quá trình tạo giọng nói với giọng [{VOICE_NAME}]...")
result_meta = {}
last_status = None
for status_msg, last_file, seg_text, q_meta in engine.generate_projects_queue_stream(
    text=raw_script,
    voice_name=voice_param,
    custom_name=fallback_name,
    auto_concat=AUTO_MERGE,
    merged_format=MERGED_FORMAT,
    cfg_scale=CFG_SCALE,
    audio_temperature=AUDIO_TEMP,
    use_voice=use_voice_flag,
    num_workers=int(NUM_WORKERS),
    result=result_meta,
):
    if status_msg and status_msg != last_status:
        last_status = status_msg
        print(f"   {status_msg}")

print(f"\n🎉 Hoàn thành toàn bộ kịch bản!")
all_folders = result_meta.get("folders", [])
for out_f in all_folders:
    if os.path.exists(out_f):
        print(f"\n📂 Thư mục xuất: {out_f}")
        for f in sorted(os.listdir(out_f)):
            fpath = os.path.join(out_f, f)
            size_kb = os.path.getsize(fpath) / 1024
            print(f"  ├── {f} ({size_kb:.1f} KB)")
